In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

/Users/jyothi/Movies/GitLearn/RAG_ENV/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jyothi/Movies/GitLearn/RAG_ENV/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("data")

Found 2 PDF files to process

Processing: js_cheet_sheet_1730678859.pdf
  ✓ Loaded 99 pages

Processing: JavaScript_Handwritten_Notes__1690868806.pdf
  ✓ Loaded 163 pages

Total documents loaded: 262


In [4]:
all_pdf_documents


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/pdf_files/js_cheet_sheet_1730678859.pdf', 'total_pages': 99, 'page': 0, 'page_label': '1', 'source_file': 'js_cheet_sheet_1730678859.pdf', 'file_type': 'pdf'}, page_content='JSM YT'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/pdf_files/js_cheet_sheet_1730678859.pdf', 'total_pages': 99, 'page': 1, 'page_label': '2', 'source_file': 'js_cheet_sheet_1730678859.pdf', 'file_type': 'pdf'}, page_content='Table of Content\nVariables\nOperators\nFunctions\nConditional Statements\nTruthy / Falsy Values \nString Escape Characters\nJavaScript Basics\nString Methods\nArray Methods\nLooping\nArray methods for looping\nString & Array Methods'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/pdf_files/js_cheet_sheet_1730678859.pdf', 'total_pages': 99, 'page': 2, 'page_label': '3', 'source_file': 'js_chee

In [7]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs
    

In [8]:
chunks=split_documents(all_pdf_documents)
chunks

Split 262 documents into 95 chunks

Example chunk:
Content: JSM YT...
Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/pdf_files/js_cheet_sheet_1730678859.pdf', 'total_pages': 99, 'page': 0, 'page_label': '1', 'source_file': 'js_cheet_sheet_1730678859.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/pdf_files/js_cheet_sheet_1730678859.pdf', 'total_pages': 99, 'page': 0, 'page_label': '1', 'source_file': 'js_cheet_sheet_1730678859.pdf', 'file_type': 'pdf'}, page_content='JSM YT'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/pdf_files/js_cheet_sheet_1730678859.pdf', 'total_pages': 99, 'page': 1, 'page_label': '2', 'source_file': 'js_cheet_sheet_1730678859.pdf', 'file_type': 'pdf'}, page_content='Table of Content\nVariables\nOperators\nFunctions\nConditional Statements\nTruthy / Falsy Values \nString Escape Characters\nJavaScript Basics\nString Methods\nArray Methods\nLooping\nArray methods for looping\nString & Array Methods'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/pdf_files/js_cheet_sheet_1730678859.pdf', 'total_pages': 99, 'page': 2, 'page_label': '3', 'source_file': 'js_chee

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

# create embeddings first
sample_texts = [
    "JavaScript basics",
    "Functions and variables in JS"
]
emb = embedding_manager.generate_embeddings(sample_texts)
# inspect embedding values
print("Type:", type(emb))
print("Shape:", emb.shape)  # (num_texts, embedding_dim)
print("First 10 values of first embedding:", emb[0][:10])
print("Min/Max:", emb.min(), emb.max())
print("Mean:", emb.mean())

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384
Generating embeddings for 2 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.84it/s]

Generated embeddings with shape: (2, 384)
Type: <class 'numpy.ndarray'>
Shape: (2, 384)
First 10 values of first embedding: [-0.03687423 -0.02811381  0.00343097  0.0453531  -0.01921049 -0.04893221
  0.07879303  0.01367892  0.01339437 -0.04660311]
Min/Max: -0.15788518 0.16421507
Mean: -0.00071679783


In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 95


In [33]:
chunks

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/pdf_files/js_cheet_sheet_1730678859.pdf', 'total_pages': 99, 'page': 0, 'page_label': '1', 'source_file': 'js_cheet_sheet_1730678859.pdf', 'file_type': 'pdf'}, page_content='JSM YT'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/pdf_files/js_cheet_sheet_1730678859.pdf', 'total_pages': 99, 'page': 1, 'page_label': '2', 'source_file': 'js_cheet_sheet_1730678859.pdf', 'file_type': 'pdf'}, page_content='Table of Content\nVariables\nOperators\nFunctions\nConditional Statements\nTruthy / Falsy Values \nString Escape Characters\nJavaScript Basics\nString Methods\nArray Methods\nLooping\nArray methods for looping\nString & Array Methods'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/pdf_files/js_cheet_sheet_1730678859.pdf', 'total_pages': 99, 'page': 2, 'page_label': '3', 'source_file': 'js_chee

In [34]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 95 texts...


Batches: 100%|██████████| 3/3 [00:01<00:00,  1.63it/s]

Generated embeddings with shape: (95, 384)
Adding 95 documents to vector store...
Successfully added 95 documents to vector store
Total documents in collection: 95


In [36]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [45]:
rag_retriever

In [54]:
rag_retriever.retrieve("ternary operator")

Retrieving documents for query: 'ternary operator'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.76it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_4e16d8c9_14',
  'content': 'Conditional Statements\nTernary Operator\ncondition ? exprIfTrue : exprIfFalse\nAn expression whose value is used as a condition.\ncondition\nAn expression which is executed if the condition is \ntruthy.\nexprIfTrue\nAn expression which is executed if the condition is \nfalsy.\nexprIfFalse',
  'metadata': {'source': 'data/pdf_files/js_cheet_sheet_1730678859.pdf',
   'creationdate': '',
   'total_pages': 99,
   'page': 15,
   'content_length': 280,
   'creator': 'PyPDF',
   'source_file': 'js_cheet_sheet_1730678859.pdf',
   'producer': 'PyPDF',
   'file_type': 'pdf',
   'doc_index': 14,
   'page_label': '16'},
  'similarity_score': 0.4082266688346863,
  'distance': 0.5917733311653137,
  'rank': 1},
 {'id': 'doc_61d531e3_9',
  'content': 'Operators\nBitwise Operators\n& AND statement\n| OR statement\n~ NOT\n^ XOR\n<< Zero fill left shift\n>> Signed right shift\n>>> Zero Fill right shift',
  'metadata': {'total_pages': 99,
   'creator': 'PyPDF',
  